# Models for Outcomes

In [18]:
import pandas as pd
import xarray as xr
import xgboost as xgb
import shap
import numpy as np
from itertools import product
from pathlib import Path
import os

In [19]:
load_path = Path(os.getcwd()).parents[0]/Path('dataset/datasets/model_ready/train.csv')
train_data = pd.read_csv(load_path, index_col='Label')

In [26]:
X = train_data[['max_ocean_SLP_gradient', 'max_landfalling_v850hPa', 'max_landfalling_omega500', 'max_IWV_ais', 'cumulative_landfalling_area', 'max_south_extent']]
snow_var = 'cumulative_snowfall_ais'

## Non-extreme outcomes

For modelling relationships between covariates and the average of the outcome variable among all ARs, we use the `xgboost` algorithm and assess variable importance via partial dependence plots at SHAP values.

In [147]:
# some basic cross validation
# for snowfall
etas = np.linspace(0.0001,0.3,11)
# units of Y are in gigatons, a loss of 1 will be really
# high, so that seems like a reasonable upper bound
gammas = np.linspace(0,1,10)
# intuition tells me we won't need to go THAT deep
max_depth = np.arange(10) + 1
# don't have a great intuition on this, mostly testing
lambdas = np.linspace(0, 2, 11)
# same
min_child_weights = np.linspace(0, 3, 11)

# since we're doing early stopping, this should be good
nrounds = 500
early_stopping_rounds = 10

tree_method = 'exact'
booster = 'gbtree'

In [148]:
hyperparams_lst = list(product(gammas, max_depth, lambdas, min_child_weights))

In [161]:
def process_hyperparam_chunk(lst):
    etas = np.array([0.001, 0.01, 0.05, 0.1, 0.2, 0.3])
    num_eta = len(etas)
    results_lst = np.zeros((len(etas)*len(lst), len(lst[0]) + 3))
    
    for i, param_set in enumerate(lst):
        for j, eta in enumerate(etas):
            params = dict(booster='gbtree',
                   eta=eta,
                   gamma=param_set[0],
                   max_depth=param_set[1],
                   reg_lambda=param_set[2],
                   min_child_weight=param_set[3],
                   tree_method=tree_method)
            dtrain = xgb.DMatrix(X, label=train_data[snow_var])
            res = xgb.cv(params,
                   dtrain,
                   nrounds,
                   nfold=5,
                   seed=12345,
                   early_stopping_rounds=early_stopping_rounds)
            results_lst[i*num_eta + j,:] = np.array(list(param_set) + [eta] + [res.shape[0]] + [float(res['test-rmse-mean'].iloc[-1])])

    return results_lst

In [171]:
chunk_lst = []
chunk_size = 1000

for i in range(0, len(hyperparams_lst), chunk_size):
    chunk_lst.append(hyperparams_lst[i:i + chunk_size])

In [174]:
pd.DataFrame(res, columns=['gamma', 'max_depth', 'reg_lambda', 'min_child_weight', 'eta', 'num_boost', 'test_rmse'])

,gamma,max_depth,reg_lambda,min_child_weight,eta,num_boost,test_rmse
0,0.0,1.0,0.0,0.3,0.001,500.0,0.495168
1,0.0,1.0,0.0,0.3,0.010,500.0,0.280186
2,0.0,1.0,0.0,0.3,0.050,304.0,0.260891
3,0.0,1.0,0.0,0.3,0.100,170.0,0.261113
4,0.0,1.0,0.0,0.3,0.200,72.0,0.261912
5,0.0,1.0,0.0,0.3,0.300,51.0,0.265174
6,0.0,1.0,0.0,0.6,0.001,500.0,0.495168
7,0.0,1.0,0.0,0.6,0.010,500.0,0.280186
8,0.0,1.0,0.0,0.6,0.050,304.0,0.260891
9,0.0,1.0,0.0,0.6,0.100,170.0,0.261113
